# YRBS 2007 Final Individual Project

**Research question:** Among U.S. high school students, is BMI percentile associated with physical activity days after adjusting for computer use, sleep, sex, and age?

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pathlib import Path

BASE = Path("..")
RAW = BASE / "data" / "raw" / "YRBS_2007.csv"

## 2. Load data

In [ ]:
df = pd.read_csv(RAW)
print(df.shape)
df.head()

## 3. Select and recode variables

The project uses BMI percentile as the outcome variable. Physical activity days is the main predictor. Computer use, sleep, sex, and age are included as control variables.

In [ ]:
cols = ["BMIPCT", "PhysicalActivity5OrMoreDays", "ComputerUse", "Sleep", "WhatIsYourSex", "HowOldAreYou", "weight"]
work = df[cols].copy().rename(columns={
    "BMIPCT":"bmi_percentile",
    "PhysicalActivity5OrMoreDays":"physical_activity_days_code",
    "ComputerUse":"computer_use_code",
    "Sleep":"sleep_code",
    "WhatIsYourSex":"sex_code",
    "HowOldAreYou":"age_code",
    "weight":"survey_weight"
})

work["physical_activity_days"] = work["physical_activity_days_code"].map({1:0,2:1,3:2,4:3,5:4,6:5,7:6,8:7})
work["computer_use_hours"] = work["computer_use_code"].map({1:0,2:0.5,3:1,4:2,5:3,6:4,7:5})
work["sleep_hours"] = work["sleep_code"].map({1:4,2:5,3:6,4:7,5:8,6:9,7:10})
work["female"] = work["sex_code"].map({1:0,2:1})
work["sex_label"] = work["female"].map({0:"Male", 1:"Female"})
work["age_years_approx"] = work["age_code"].map({1:12,2:13,3:14,4:15,5:16,6:17,7:18})

clean = work.dropna(subset=["bmi_percentile", "physical_activity_days", "computer_use_hours", "sleep_hours", "female", "age_years_approx"]).copy()
clean.shape

## 4. Save cleaned data

In [ ]:
processed_dir = BASE / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
clean.to_csv(processed_dir / "yrbs_2007_cleaned_regression.csv", index=False)

## 5. Descriptive statistics

In [ ]:
summary_stats = clean[["bmi_percentile", "physical_activity_days", "computer_use_hours", "sleep_hours", "female", "age_years_approx"]].describe().T
summary_stats

In [ ]:
tables_dir = BASE / "outputs" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
summary_stats.to_csv(tables_dir / "descriptive_statistics.csv")

## 6. Figure: Mean BMI percentile by physical activity days

In [ ]:
figures_dir = BASE / "outputs" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
by_pa = clean.groupby("physical_activity_days")["bmi_percentile"].agg(["count", "mean", "std"]).reset_index()
by_pa.to_csv(tables_dir / "bmi_by_physical_activity_days.csv", index=False)

plt.figure(figsize=(8,5))
plt.bar(by_pa["physical_activity_days"], by_pa["mean"], yerr=1.96*by_pa["std"]/np.sqrt(by_pa["count"]), capsize=4)
plt.xlabel("Physical activity days in past 7 days")
plt.ylabel("Mean BMI percentile")
plt.title("Mean BMI Percentile by Physical Activity Days")
plt.tight_layout()
plt.savefig(figures_dir / "mean_bmi_by_activity.png", dpi=200)
plt.show()

## 7. Regression analysis

In [ ]:
X = clean[["physical_activity_days", "computer_use_hours", "sleep_hours", "female", "age_years_approx"]]
X = sm.add_constant(X)
y = clean["bmi_percentile"]
model = sm.OLS(y, X).fit()
print(model.summary())

## 8. Save regression table

In [ ]:
coef_table = pd.DataFrame({
    "Variable": ["Intercept", "Physical activity days", "Computer use hours", "Sleep hours", "Female (vs male)", "Age (approx years)"],
    "Coefficient": model.params.values,
    "Std. Error": model.bse.values,
    "t value": model.tvalues.values,
    "p value": model.pvalues.values,
    "CI Lower": model.conf_int()[0].values,
    "CI Upper": model.conf_int()[1].values,
})
coef_table.to_csv(tables_dir / "regression_results.csv", index=False)
coef_table

## 9. Figure: Coefficient plot

In [ ]:
ct = coef_table.iloc[1:].copy()
plt.figure(figsize=(8,4.8))
ypos = np.arange(len(ct))
plt.errorbar(ct["Coefficient"], ypos, xerr=[ct["Coefficient"]-ct["CI Lower"], ct["CI Upper"]-ct["Coefficient"]], fmt="o", capsize=4)
plt.axvline(0, linewidth=1)
plt.yticks(ypos, ct["Variable"])
plt.xlabel("OLS coefficient, 95% CI")
plt.title("Regression Coefficients Predicting BMI Percentile")
plt.tight_layout()
plt.savefig(figures_dir / "regression_coefficients.png", dpi=200)
plt.show()

## 10. Interpretation

The coefficient for physical activity days is negative and statistically significant in the OLS model. However, the effect size is small, and R-squared is low. Therefore, the result should be interpreted as a weak association, not a causal effect.